# Semantic Layer - Admin Notebook

Operations notebook for managing Neptune and Ontop backends.

| Section | Action |
|---------|--------|
| **Setup** | Configure endpoints via workshop.py |
| **Neptune - Summary** | Class/property/instance counts |
| **Neptune - Reset** | ⚠️ Drop all triples from default graph |
| **Ontop - Summary** | Triple counts from the VKG |


---
### Setup


In [ ]:
ONTOP_ENDPOINT = ""  # Set by lifecycle config


In [ ]:
%graph_notebook_config --store-to NOTEBOOK_CONFIG --silent


In [ ]:
from workshop import *
import json
configure(
    ontop_endpoint=ONTOP_ENDPOINT,
    neptune_config=json.loads(NOTEBOOK_CONFIG)
)
print(f"Neptune: {NEPTUNE_HOST}")
print(f"Ontop:   {ONTOP_URL}")


---
## Neptune - Summary

Counts triples, classes, properties, and instances per class in Neptune.


In [ ]:
# Neptune graph summary
rows = sparql_query("SELECT (COUNT(*) AS ?count) WHERE { ?s ?p ?o }")
total = int(rows[0]["count"])
print(f"Total triples: {total}")

if total > 0:
    print("\nInstances per class:")
    rows = sparql_query("""
        PREFIX owl: <http://www.w3.org/2002/07/owl#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT ?class ?label (COUNT(?instance) AS ?count)
        WHERE {
            ?class a owl:Class .
            OPTIONAL { ?class rdfs:label ?label }
            OPTIONAL { ?instance a ?class }
        }
        GROUP BY ?class ?label
        ORDER BY ?label
    """)
    for r in rows:
        label = r.get("label", r["class"].split("#")[-1])
        print(f"  {label}: {r['count']}")


---
## Neptune - Reset (Drop All Triples)

⚠️ **This deletes ALL triples from the default graph.** Run the Summary cell above first to confirm what's there.


In [ ]:
# Drop all triples from Neptune
rows = sparql_query("SELECT (COUNT(*) AS ?count) WHERE { ?s ?p ?o }")
before = int(rows[0]["count"])
print(f"Triples before: {before}")

if before > 0:
    status = neptune_update("DROP ALL")
    rows = sparql_query("SELECT (COUNT(*) AS ?count) WHERE { ?s ?p ?o }")
    after = int(rows[0]["count"])
    print(f"Triples after:  {after}")
    print(f"Dropped {before - after} triples")
else:
    print("Graph is already empty.")


---
## Ontop VKG - Summary

Same summary queries against the Ontop Virtual Knowledge Graph endpoint.
Note: Ontop exposes relational data as RDF - counts reflect the OBDA mappings.


In [ ]:
# Ontop graph summary
rows = ontop_query("SELECT (COUNT(*) AS ?count) WHERE { ?s ?p ?o }")
total = int(rows[0]["count"])
print(f"Total triples (virtual): {total}")

if total > 0:
    print("\nInstances per class:")
    rows = ontop_query("""
        PREFIX : <http://example.org/insurance#>
        SELECT ?class (COUNT(?instance) AS ?count)
        WHERE {
            ?instance a ?class .
            FILTER(STRSTARTS(STR(?class), STR(:)))
        }
        GROUP BY ?class
        ORDER BY ?class
    """)
    for r in rows:
        cls = r["class"].split("#")[-1]
        print(f"  {cls}: {r['count']}")
